# Survival Fixed Dataset Analysis

Notebook for fixed observed-dataset survival runs. It mirrors `fixed_dataset_analysis.ipynb`, but computes replicate-level C-index from stored survival risk scores.


In [ ]:
from pathlib import Path

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'analysis' else cwd

DATASET_NAME = 'mmColorectal'
LABEL_NAME = 'OS'
RETRAIN_OUTER = False
USE_ENSEMBLE = True

RESULTS_TAG = f'{DATASET_NAME}_{LABEL_NAME}' if str(LABEL_NAME).strip() else DATASET_NAME
RESULTS_ROOT = PROJECT_ROOT / 'results' / RESULTS_TAG / 'training_runs'
if not RESULTS_ROOT.exists():
    raise FileNotFoundError(f'Results root does not exist: {RESULTS_ROOT}')
if not RESULTS_ROOT.is_dir():
    raise NotADirectoryError(f'Expected a directory at: {RESULTS_ROOT}')

RETRAIN_TAG = f"retrain{str(bool(RETRAIN_OUTER)).lower()}"
PREDICTION_TAG = 'ensemble' if bool(USE_ENSEMBLE) else 'inner_models'
OUTPUT_DIR = (
    (cwd / 'survival_fixed_dataset_analysis_outputs') if cwd.name == 'analysis' else (cwd / 'analysis' / 'survival_fixed_dataset_analysis_outputs')
) / RESULTS_TAG / RETRAIN_TAG / PREDICTION_TAG
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIGURES_DIR = OUTPUT_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_ROOT


In [ ]:
import sys
import importlib
import pandas as pd

ANALYSIS_DIR = cwd if cwd.name == 'analysis' else (cwd / 'analysis')
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))

import results_analysis as ra
ra = importlib.reload(ra)

HAVE_MPL = ra.HAVE_MPL
resolve_requested_model_names = ra.resolve_requested_model_names
load_all_survival_test_predictions = ra.load_all_survival_test_predictions
expand_inner_model_survival_predictions = ra.expand_inner_model_survival_predictions
aggregate_member_patient_survival_predictions = ra.aggregate_member_patient_survival_predictions
build_replicate_cindex_table = ra.build_replicate_cindex_table
cindex_replicates_to_auc_compatible = ra.cindex_replicates_to_auc_compatible
build_level1_summary = ra.build_level1_summary
build_fixed_dataset_method_summary = ra.build_fixed_dataset_method_summary
compute_fixed_dataset_global_friedman = ra.compute_fixed_dataset_global_friedman
compute_level2_pairwise_tests = ra.compute_level2_pairwise_tests
build_fixed_dataset_method_significance_summary = ra.build_fixed_dataset_method_significance_summary
rename_auc_outputs_for_cindex = ra.rename_auc_outputs_for_cindex
plot_fixed_dataset_auc_violins = ra.plot_fixed_dataset_auc_violins
plot_level3_pairwise_condition_matrices = ra.plot_level3_pairwise_condition_matrices


In [ ]:
REQUESTED_MODEL_NAMES = resolve_requested_model_names(
    results_root=RESULTS_ROOT,
    dataset_name=DATASET_NAME,
    retrain_outer=RETRAIN_OUTER,
    results_mode='fixed_dataset',
)
if not REQUESTED_MODEL_NAMES:
    raise ValueError('No model folders were detected for the requested dataset / retrain flag / fixed-dataset mode.')

raw_predictions_df, missing_prediction_files = load_all_survival_test_predictions(
    results_root=RESULTS_ROOT,
    dataset_name=DATASET_NAME,
    model_names=REQUESTED_MODEL_NAMES,
    retrain_outer=RETRAIN_OUTER,
    results_mode='fixed_dataset',
    use_ensemble=USE_ENSEMBLE,
)
if raw_predictions_df.empty:
    raise ValueError('No survival test_predictions.csv files with usable risk scores were found.')

member_prediction_df = expand_inner_model_survival_predictions(raw_predictions_df, use_ensemble=USE_ENSEMBLE)
member_patient_df = aggregate_member_patient_survival_predictions(member_prediction_df)
replicate_cindex_df = build_replicate_cindex_table(member_patient_df)
replicate_metric_df = cindex_replicates_to_auc_compatible(replicate_cindex_df)

fixed_level1_internal_df = build_level1_summary(replicate_metric_df)
fixed_method_summary_internal_df = build_fixed_dataset_method_summary(replicate_metric_df)
fixed_global_friedman_df = compute_fixed_dataset_global_friedman(replicate_metric_df)
fixed_pairwise_internal_df = compute_level2_pairwise_tests(fixed_level1_internal_df, replicate_metric_df)
fixed_significant_internal_df = fixed_pairwise_internal_df.loc[fixed_pairwise_internal_df['significant_fdr_0p05']].reset_index(drop=True)
fixed_method_table_internal_df = build_fixed_dataset_method_significance_summary(
    fixed_method_summary_internal_df,
    fixed_pairwise_internal_df,
)

fixed_level1_df = rename_auc_outputs_for_cindex(fixed_level1_internal_df)
fixed_method_summary_df = rename_auc_outputs_for_cindex(fixed_method_summary_internal_df)
fixed_pairwise_df = rename_auc_outputs_for_cindex(fixed_pairwise_internal_df)
fixed_significant_df = rename_auc_outputs_for_cindex(fixed_significant_internal_df)
fixed_method_table_df = rename_auc_outputs_for_cindex(fixed_method_table_internal_df)

paths = {
    'replicate_cindex_table.csv': replicate_cindex_df,
    'fixed_dataset_method_condition_summary.csv': fixed_level1_df,
    'fixed_dataset_method_summary.csv': fixed_method_summary_df,
    'fixed_dataset_global_friedman.csv': fixed_global_friedman_df,
    'fixed_dataset_pairwise_wilcoxon.csv': fixed_pairwise_df,
    'fixed_dataset_pairwise_significant.csv': fixed_significant_df,
    'fixed_dataset_method_table.csv': fixed_method_table_df,
}
for file_name, df in paths.items():
    path = OUTPUT_DIR / file_name
    df.to_csv(path, index=False)
    print('Saved:', path)

print('Missing prediction files:', len(missing_prediction_files))


## Fixed survival ranking

Method ranking by mean C-index, with bootstrap CIs and fixed-dataset pairwise tests.


In [ ]:
display(fixed_method_table_df)
print('Global Friedman test:')
display(fixed_global_friedman_df)
print('Significant pairwise comparisons:')
display(fixed_significant_df)

model_unit_label = 'ensemble predictions' if bool(USE_ENSEMBLE) else ('outer models' if bool(RETRAIN_OUTER) else 'inner models')
if HAVE_MPL:
    plot_fixed_dataset_auc_violins(
        replicate_auc_df=replicate_metric_df,
        method_summary_df=fixed_method_table_internal_df,
        title=f'Fixed-dataset replicate C-index distributions | {DATASET_NAME} | {LABEL_NAME} | {model_unit_label}',
        figures_dir=FIGURES_DIR,
        file_name=f'fixed_dataset_cindex_violins_{RESULTS_TAG}_{RETRAIN_TAG}_{PREDICTION_TAG}.png',
    )
    plot_level3_pairwise_condition_matrices(
        level2_pairwise_df=fixed_pairwise_internal_df,
        title=f'Pairwise Wilcoxon Signed-Rank + FDR | fixed dataset | ordered by mean C-index | {DATASET_NAME} | {LABEL_NAME}',
        figures_dir=FIGURES_DIR,
        file_name=f'fixed_dataset_pairwise_cindex_matrix_{RESULTS_TAG}_{RETRAIN_TAG}_{PREDICTION_TAG}.png',
    )
else:
    print('Matplotlib not available; skipping plots.')
